In [1]:
# ------------------------------------------------------------
# CMA-Brain2Text: Contrastive Metadata Alignment
# A NOVEL EEG-to-Text model with privileged visual guidance
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader, random_split
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
from collections import Counter
from torch.amp import autocast as amp_autocast, GradScaler as amp_GradScaler
import sacrebleu
import gc


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch_cluster/nearest.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  import scipy.cluster


In [8]:
!pip install tables


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 1.3 MB/s eta 0:00:001.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 1.9 MB/s eta 0:00:00 MB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.2/528.2 kB 1.4 MB/s eta 0:00:00? eta -:--:--

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [9]:

import pandas as pd

# read HDF5 file
df = pd.read_hdf("/home/poorna/data/eeg_dataset_with_qwen.h5")

# print column names
print(df.columns.tolist())


ValueError: Dataset(s) incompatible with Pandas data types, not table, or no datasets found in HDF5 file.

In [4]:
# ------------------------------------------------------------
# CONFIG (EDIT THESE)
# ------------------------------------------------------------
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"           # Your HDF5 file
MODEL_SAVE_PATH = "eeg-meta-text-transformer-v5-model.pt"
BATCH_SIZE = 16
TRAIN_PCT = 0.8
VAL_PCT = 0.1
D_MODEL = 256
NUM_COLORS = 12
NUM_OBJECTS = 90
TEXT_VOCAB_SIZE = 32000
PAD_ID = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 1. DATASET
# ------------------------------------------------------------
def collate_fn(batch):
    eeg, meta, text = zip(*batch)
    eeg = torch.nn.utils.rnn.pad_sequence(eeg, batch_first=True, padding_value=0.0)
    meta = torch.stack(meta)
    text = torch.nn.utils.rnn.pad_sequence(text, batch_first=True, padding_value=PAD_ID)
    return eeg, meta, text
# ------------------------------------------------------------
# 1. FIXED DATASET — AUTO-DETECT KEYS
# ------------------------------------------------------------
class EEGMetaTextDataset(Dataset):
    def __init__(self, h5_path):
        self.file = h5py.File(h5_path, 'r')
        
        # --- PRINT WHAT'S IN THE FILE ---
        print("HDF5 file keys:", list(self.file.keys()))
        
        # --- AUTO-DETECT EEG, META, TEXT ---
        if 'eeg' in self.file:
            self.eeg = self.file['eeg']
        else:
            raise KeyError("No 'eeg' dataset found!")
        
        if 'meta' in self.file:
            self.meta = self.file['meta']
        elif 'metadata' in self.file:
            self.meta = self.file['metadata']
            print("Using 'metadata' as meta")
        else:
            raise KeyError("No 'meta' or 'metadata' dataset found!")
        
        if 'text' in self.file:
            self.text = self.file['text']
        elif 'caption' in self.file:
            self.text = self.file['caption']
            print("Using 'caption' as text")
        else:
            raise KeyError("No 'text' or 'caption' dataset found!")
            
        print(f"Dataset loaded: {len(self)} samples")
        print(f"   EEG shape: {self.eeg[0].shape}")
        print(f"   Meta shape: {self.meta[0].shape}")
        print(f"   Text length: {len(self.text[0])}")

    def __len__(self):
        return len(self.eeg)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.eeg[idx], dtype=torch.float32),
            torch.tensor(self.meta[idx], dtype=torch.float32),
            torch.tensor(self.text[idx], dtype=torch.long)
        )
# ------------------------------------------------------------
# 2. FULLY CONNECTED GRAPH (62 EEG channels)
# ------------------------------------------------------------
def fully_connected_graph(n_nodes=62):
    idx = torch.combinations(torch.arange(n_nodes), r=2)
    edge_index = torch.cat([idx, idx.flip(1)], dim=0).t().contiguous()
    edge_weight = torch.ones(edge_index.size(1))
    return edge_index.to(DEVICE), edge_weight.to(DEVICE)

# ------------------------------------------------------------
# 3. METADATA ENCODER (PRIVILEGED TEACHER)
# ------------------------------------------------------------
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, dim=256):
        super().__init__()
        self.color_emb = nn.Embedding(num_colors, 64)
        self.obj_proj = nn.Sequential(
            nn.Linear(num_objects, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, dim)
        )
        self.norm = nn.LayerNorm(dim)
    def forward(self, meta):
        c = self.color_emb(meta[:, 0].long())
        o = self.obj_proj(meta[:, 1:].float())
        return self.norm(c + o)  # (B, dim)

# ------------------------------------------------------------
# 4. EEG ENCODER + [CLS] TOKEN
# ------------------------------------------------------------
class EEGEncoder(nn.Module):
    def __init__(self, d_model=256, n_layers=4, n_heads=8):
        super().__init__()
        self.gcn = GCNConv(62, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=1024,
            dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_emb = nn.Parameter(torch.randn(1, 1000, d_model))  # max time steps

    def forward(self, eeg, edge_idx, edge_w):
        B, C, T = eeg.shape
        # GCN on time-flattened
        x = eeg.permute(0, 2, 1).reshape(B*T, C)
        x = self.gcn(x, edge_idx, edge_w).view(B, T, -1)
        x = x + self.pos_emb[:, :T, :]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, T+1, D)
        return self.transformer(x)     # (B, T+1, D)

# ------------------------------------------------------------
# 5. META HEAD (for auxiliary supervision)
# ------------------------------------------------------------
class MetaHead(nn.Module):
    def __init__(self, d_model, num_colors, num_objects):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.5)
        )
        self.color_head = nn.Linear(128, num_colors)
        self.obj_head = nn.Linear(128, num_objects)
    def forward(self, x):
        x = self.net(x)
        return torch.cat([self.color_head(x), self.obj_head(x)], dim=1)

# ------------------------------------------------------------
# 6. TEXT DECODER
# ------------------------------------------------------------
class TextDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, n_heads=8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=1024,
            dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.out = nn.Linear(d_model, vocab_size)
    def forward(self, tgt, memory):
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.emb(tgt)
        x = self.transformer(x, memory, tgt_mask=mask)
        return self.out(x)

# ------------------------------------------------------------
# 7. FULL MODEL: CMA-Brain2Text
# ------------------------------------------------------------
class CMA_Brain2Text(nn.Module):
    def __init__(self, vocab_size, num_colors, num_objects):
        super().__init__()
        self.enc = EEGEncoder(D_MODEL)
        self.meta_head = MetaHead(D_MODEL, num_colors, num_objects)
        self.meta_enc = MetadataEncoder(num_colors, num_objects, D_MODEL)
        self.proj_eeg = nn.Linear(D_MODEL, D_MODEL)    # EEG → contrastive space
        self.proj_meta = nn.Linear(D_MODEL, D_MODEL)  # Meta → contrastive space
        self.dec = TextDecoder(vocab_size, D_MODEL)

    def forward(self, eeg, meta, tgt, edge_idx, edge_w, stage=1, tf_ratio=1.0):
        """
        stage=1: Pre-train meta head
        stage=2: Contrastive alignment
        stage=3: Full text generation
        """
        enc = self.enc(eeg, edge_idx, edge_w)      # (B, T+1, D)
        cls = enc[:, 0]                            # (B, D)
        meta_pred = self.meta_head(cls)            # (B, 12+90)
        meta_gt = self.meta_enc(meta)              # (B, D)

        # --- STAGE 1: Meta prediction loss ---
        if stage == 1:
            return meta_pred, None

        # --- STAGE 2: Contrastive alignment ---
        if stage == 2:
            eeg_proj = F.normalize(self.proj_eeg(cls), dim=-1)
            meta_proj = F.normalize(self.proj_meta(meta_gt), dim=-1)
            logits = eeg_proj @ meta_proj.T * 10.0
            labels = torch.arange(eeg.shape[0], device=eeg.device)
            loss_cont = F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)
            return meta_pred, loss_cont

        # --- STAGE 3: Text generation (ONLY EEG) ---
        if stage == 3:
            # Teacher forcing
            if self.training and tf_ratio > 0:
                shifted_tgt = torch.cat([
                    torch.full((tgt.size(0), 1), 1, device=tgt.device),  # <bos>
                    tgt[:, :-1]
                ], dim=1)
                if torch.rand(1).item() < tf_ratio:
                    tgt = shifted_tgt
            dec_out = self.dec(tgt, enc)
            return dec_out, meta_pred, None

# ------------------------------------------------------------
# 8. GENERATE FUNCTION
# ------------------------------------------------------------
def generate(model, eeg, edge_idx, edge_w, tokenizer, max_len=20):
    model.eval()
    with torch.no_grad():
        bos_id = tokenizer.bos_token_id or 1
        ids = torch.tensor([[bos_id]], device=eeg.device)
        for _ in range(max_len):
            logits, _, _ = model(eeg, None, ids, edge_idx, edge_w, stage=3, tf_ratio=0.0)
            next_id = logits[0, -1].argmax(-1).unsqueeze(0)
            ids = torch.cat([ids, next_id], dim=1)
            if next_id.item() == tokenizer.eos_token_id:
                break
        return tokenizer.decode(ids.squeeze(0).cpu().tolist(), skip_special_tokens=True)


In [5]:


# ------------------------------------------------------------
# 9. MAIN: 3-STAGE TRAINING WITH PROGRESS
# ------------------------------------------------------------
if __name__ == "__main__":
    print("CMA-BRAIN2TEXT: CONTRASTIVE METADATA ALIGNMENT")
    print("="*60)

    # --- Data ---
    ds = EEGMetaTextDataset(H5_FILE_PATH)
    N = len(ds)
    trN = int(N*TRAIN_PCT); valN = int(N*VAL_PCT); teN = N-trN-valN
    tr, va, te = random_split(ds, [trN, valN, teN], generator=torch.Generator().manual_seed(42))
    trL = DataLoader(tr, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    vaL = DataLoader(va, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    teL = DataLoader(te, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    edge_idx, edge_w = fully_connected_graph()

    # --- Model ---
    model = CMA_Brain2Text(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS).to(DEVICE)

    # --- Losses ---
    color_counts = Counter(m[:, 0].long().cpu().item() for _, m, _ in tr)
    weights = [1.0 / color_counts.get(i, 1) for i in range(NUM_COLORS)]
    color_weights = torch.tensor(weights, device=DEVICE)
    cc = nn.CrossEntropyLoss(weight=color_weights, label_smoothing=0.1)
    oc = nn.BCEWithLogitsLoss()
    tc = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

    # -------------------------------------------------
    # STAGE 1: PRE-TRAIN META HEAD
    # -------------------------------------------------
    print("\nSTAGE 1: Pre-training Meta Head (Decoder Frozen)")
    for p in model.dec.parameters(): p.requires_grad = False
    opt1 = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-2)

    for epoch in range(5):
        model.train()
        total_loss = 0
        prog = tqdm(trL, desc=f"S1 | Epoch {epoch+1}/5")
        for eeg, meta, _ in prog:
            eeg, meta = eeg.to(DEVICE), meta.to(DEVICE)
            opt1.zero_grad()
            with amp_autocast('cuda'):
                mp, _ = model(eeg, meta, None, edge_idx, edge_w, stage=1)
                loss = cc(mp[:, :12], meta[:, 0].long()) + 2.0 * oc(mp[:, 12:], meta[:, 1:].float())
            scaler = amp_GradScaler('cuda')
            scaler.scale(loss).backward()
            scaler.step(opt1)
            scaler.update()
            total_loss += loss.item()
            prog.set_postfix(loss=f"{loss.item():.3f}")
        print(f"→ S1 Epoch {epoch+1} | Avg Loss: {total_loss/len(trL):.4f}")

    # -------------------------------------------------
    # STAGE 2: CONTRASTIVE ALIGNMENT
    # -------------------------------------------------
    print("\nSTAGE 2: Contrastive Metadata Alignment (CMA)")
    for p in model.dec.parameters(): p.requires_grad = False
    opt2 = AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)

    for epoch in range(10):
        model.train()
        total_cont = 0
        prog = tqdm(trL, desc=f"S2 | Epoch {epoch+1}/10")
        for eeg, meta, _ in prog:
            eeg, meta = eeg.to(DEVICE), meta.to(DEVICE)
            opt2.zero_grad()
            with amp_autocast('cuda'):
                _, loss_cont = model(eeg, meta, None, edge_idx, edge_w, stage=2)
            scaler.scale(loss_cont).backward()
            scaler.step(opt2)
            scaler.update()
            total_cont += loss_cont.item()
            prog.set_postfix(cont_loss=f"{loss_cont.item():.3f}")
        print(f"→ S2 Epoch {epoch+1} | Avg Cont Loss: {total_cont/len(trL):.4f}")

    # -------------------------------------------------
    # STAGE 3: FULL TEXT GENERATION
    # -------------------------------------------------
    print("\nSTAGE 3: Full Text Generation (Decoder Unlocked)")
    for p in model.parameters(): p.requires_grad = True
    opt3 = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    sched = ReduceLROnPlateau(opt3, 'min', factor=0.5, patience=3)
    tf_sched = np.concatenate([np.ones(30), np.linspace(1.0, 0.0, 30)])
    best = float('inf')

    for epoch in range(1, 31):
        tf = tf_sched[min(epoch-1, len(tf_sched)-1)]
        model.train()
        total_loss = 0
        prog = tqdm(trL, desc=f"S3 | Epoch {epoch}/30 | TF {tf:.2f}")
        for eeg, meta, txt in prog:
            eeg, meta, txt = eeg.to(DEVICE), meta.to(DEVICE), txt.to(DEVICE)
            opt3.zero_grad()
            with amp_autocast('cuda'):
                logits, mp, _ = model(eeg, meta, txt, edge_idx, edge_w, stage=3, tf_ratio=tf)
                loss_text = tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1))
                loss_meta = cc(mp[:, :12], meta[:, 0].long()) + oc(mp[:, 12:], meta[:, 1:].float())
                loss = loss_text + 0.5 * loss_meta
            scaler.scale(loss).backward()
            scaler.step(opt3)
            scaler.update()
            total_loss += loss.item()
            prog.set_postfix(loss=f"{loss.item():.3f}")
        val_loss = 0
        model.eval()
        with torch.no_grad():
            for eeg, meta, txt in vaL:
                eeg, meta, txt = eeg.to(DEVICE), meta.to(DEVICE), txt.to(DEVICE)
                logits, mp, _ = model(eeg, meta, txt, edge_idx, edge_w, stage=3)
                val_loss += (tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1)) + 0.5 * (
                    cc(mp[:, :12], meta[:, 0].long()) + oc(mp[:, 12:], meta[:, 1:].float())
                )).item()
        val_loss /= len(vaL)
        sched.step(val_loss)
        print(f"→ S3 Epoch {epoch} | Train: {total_loss/len(trL):.4f} | Val: {val_loss:.4f}")
        if val_loss < best:
            best = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("   [BEST MODEL SAVED]")

    

CMA-BRAIN2TEXT: CONTRASTIVE METADATA ALIGNMENT
HDF5 file keys: ['eeg', 'input_ids', 'metadata']
Using 'metadata' as meta


KeyError: "No 'text' or 'caption' dataset found!"

In [ ]:
# -------------------------------------------------
    # FINAL TEST + GENERATION
    # -------------------------------------------------
    print("\nFINAL TEST & BRAIN-TO-TEXT GENERATION")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    test_loss, test_cacc = 0, 0
    with torch.no_grad():
        for eeg, meta, txt in teL:
            eeg, meta, txt = eeg.to(DEVICE), meta.to(DEVICE), txt.to(DEVICE)
            logits, mp, _ = model(eeg, meta, txt, edge_idx, edge_w, stage=3)
            test_loss += tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1)).item()
            test_cacc += (mp[:, :12].argmax(1) == meta[:, 0].long()).float().mean().item()
    print(f"TEST → Loss: {test_loss/len(teL):.4f} | ColorAcc: {test_cacc/len(teL):.3f}")

    print("\nSAMPLE BRAIN-TO-TEXT:")
    with torch.no_grad():
        for i, (eeg, _, txt) in enumerate(teL):
            if i >= 3: break
            eeg = eeg.to(DEVICE)
            pred = generate(model, eeg[0:1], edge_idx, edge_w, tokenizer)
            true = tokenizer.decode(txt[0].tolist(), skip_special_tokens=True)
            print(f"EEG → {pred}")
            print(f"True: {true}\n")

In [14]:
# ------------------------------------------------------------
# CMA-BRAIN2TEXT: FINAL CODE (WITH YOUR ORIGINAL DATASET)
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader, random_split
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
from collections import Counter
from torch.amp import autocast as amp_autocast, GradScaler as amp_GradScaler
import gc

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"

MODEL_SAVE_PATH = 'eeg-meta-text-transformer-v5-model.pt'
BATCH_SIZE = 16
TRAIN_PCT = 0.8
VAL_PCT = 0.1
D_MODEL = 256
NUM_COLORS = 12
NUM_OBJECTS = 90
TEXT_VOCAB_SIZE = 32000
PAD_ID = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 1. YOUR ORIGINAL DATASET (UNCHANGED + PRINT INFO)
# ------------------------------------------------------------
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
            print(f"Dataset: {self.n_samples} samples")
            print(f"   EEG shape: {f['eeg'].shape[1:]}")
            print(f"   Meta shape: {f['metadata'].shape[1:]}")
            print(f"   Text shape: {f['input_ids'].shape[1:]}")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
       
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))      # (62, T)
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32')) # (91,)
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))  # (L,)

        return eeg, meta, text

# ------------------------------------------------------------
# 2. COLLATE FN (PAD TIME & TEXT)
# ------------------------------------------------------------
def collate_fn(batch):
    eeg, meta, text = zip(*batch)
    eeg = torch.nn.utils.rnn.pad_sequence(eeg, batch_first=True, padding_value=0.0)
    meta = torch.stack(meta)
    text = torch.nn.utils.rnn.pad_sequence(text, batch_first=True, padding_value=PAD_ID)
    return eeg, meta, text

# ------------------------------------------------------------
# 3. GRAPH
# ------------------------------------------------------------
def fully_connected_graph(n_nodes=62):
    idx = torch.combinations(torch.arange(n_nodes), r=2)
    edge_index = torch.cat([idx, idx.flip(1)], dim=0).t().contiguous()
    edge_weight = torch.ones(edge_index.size(1))
    return edge_index.to(DEVICE), edge_weight.to(DEVICE)

# ------------------------------------------------------------
# 4. METADATA ENCODER (PRIVILEGED TEACHER)
# ------------------------------------------------------------
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, dim=256):
        super().__init__()
        self.color_emb = nn.Embedding(num_colors, dim)   # ← dim = 256
        self.obj_proj = nn.Sequential(
            nn.Linear(num_objects, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, dim)  # ← output dim = 256
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, meta):
        c = self.color_emb(meta[:, 0].long())           # (B, 256)
        o = self.obj_proj(meta[:, 1:].float())          # (B, 256)
        return self.norm(c + o)                         # (B, 256)
# ------------------------------------------------------------
# 5. EEG ENCODER + META HEAD
# ------------------------------------------------------------
class EEGEncoder(nn.Module):
    def __init__(self, d_model=256, n_layers=4, n_heads=8):
        super().__init__()
        self.gcn = GCNConv(62, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=1024,
            dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_emb = nn.Parameter(torch.randn(1, 1000, d_model))

    def forward(self, eeg, edge_idx, edge_w):
        B, C, T = eeg.shape
        x = eeg.permute(0, 2, 1).reshape(B*T, C)
        x = self.gcn(x, edge_idx, edge_w).view(B, T, -1)
        x = x + self.pos_emb[:, :T, :]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        return self.transformer(x)

class MetaHead(nn.Module):
    def __init__(self, d_model, num_colors, num_objects):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.5)
        )
        self.color_head = nn.Linear(128, num_colors)
        self.obj_head = nn.Linear(128, num_objects)
    def forward(self, x):
        x = self.net(x)
        return torch.cat([self.color_head(x), self.obj_head(x)], dim=1)

# ------------------------------------------------------------
# 6. TEXT DECODER
# ------------------------------------------------------------
class TextDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, n_heads=8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=1024,
            dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.out = nn.Linear(d_model, vocab_size)
    def forward(self, tgt, memory):
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.emb(tgt)
        x = self.transformer(x, memory, tgt_mask=mask)
        return self.out(x)

# ------------------------------------------------------------
# 7. FULL MODEL: CMA-Brain2Text
# ------------------------------------------------------------
class CMA_Brain2Text(nn.Module):
    def __init__(self, vocab_size, num_colors, num_objects):
        super().__init__()
        self.enc = EEGEncoder(D_MODEL)
        self.meta_head = MetaHead(D_MODEL, num_colors, num_objects)
        self.meta_enc = MetadataEncoder(num_colors, num_objects, D_MODEL)
        self.proj_eeg = nn.Linear(D_MODEL, D_MODEL)
        self.proj_meta = nn.Linear(D_MODEL, D_MODEL)
        self.dec = TextDecoder(vocab_size, D_MODEL)

    def forward(self, eeg, meta, tgt, edge_idx, edge_w, stage=1, tf_ratio=1.0):
        enc = self.enc(eeg, edge_idx, edge_w)
        cls = enc[:, 0]
        meta_pred = self.meta_head(cls)
        meta_gt = self.meta_enc(meta)

        if stage == 1:
            return meta_pred, None

        if stage == 2:
            eeg_proj = F.normalize(self.proj_eeg(cls), dim=-1)
            meta_proj = F.normalize(self.proj_meta(meta_gt), dim=-1)
            logits = eeg_proj @ meta_proj.T * 10.0
            labels = torch.arange(eeg.shape[0], device=eeg.device)
            loss_cont = F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)
            return meta_pred, loss_cont

        if stage == 3:
            if self.training and tf_ratio > 0:
                shifted_tgt = torch.cat([
                    torch.full((tgt.size(0), 1), 1, device=tgt.device),
                    tgt[:, :-1]
                ], dim=1)
                if torch.rand(1).item() < tf_ratio:
                    tgt = shifted_tgt
            dec_out = self.dec(tgt, enc)
            return dec_out, meta_pred, None

# ------------------------------------------------------------
# 8. GENERATE
# ------------------------------------------------------------
def generate(model, eeg, edge_idx, edge_w, tokenizer, max_len=20):
    model.eval()
    with torch.no_grad():
        bos_id = tokenizer.bos_token_id or 1
        ids = torch.tensor([[bos_id]], device=eeg.device)
        for _ in range(max_len):
            logits, _, _ = model(eeg, None, ids, edge_idx, edge_w, stage=3, tf_ratio=0.0)
            next_id = logits[0, -1].argmax(-1).unsqueeze(0)
            ids = torch.cat([ids, next_id], dim=1)
            if next_id.item() == tokenizer.eos_token_id:
                break
        return tokenizer.decode(ids.squeeze(0).cpu().tolist(), skip_special_tokens=True)



In [15]:
# ------------------------------------------------------------
# 9. MAIN: 3-STAGE TRAINING
# ------------------------------------------------------------
if __name__ == "__main__":
    print("CMA-BRAIN2TEXT: NOVEL CONTRASTIVE EEG-TO-TEXT")
    print("="*60)

    # --- Data ---
    ds = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(ds)
    trN = int(N*TRAIN_PCT); valN = int(N*VAL_PCT); teN = N-trN-valN
    tr, va, te = random_split(ds, [trN, valN, teN], generator=torch.Generator().manual_seed(42))
    trL = DataLoader(tr, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    vaL = DataLoader(va, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    teL = DataLoader(te, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    edge_idx, edge_w = fully_connected_graph()

    model = CMA_Brain2Text(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS).to(DEVICE)

    # --- Losses ---
    # --- CORRECT: m[0] instead of m[:, 0] ---
    color_counts = Counter(int(m[0].item()) for _, m, _ in tr)
    weights = [1.0 / color_counts.get(i, 1) for i in range(NUM_COLORS)]
    color_weights = torch.tensor(weights, device=DEVICE)
    cc = nn.CrossEntropyLoss(weight=color_weights, label_smoothing=0.1)
    oc = nn.BCEWithLogitsLoss()
    tc = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

    # -------------------------------------------------
    # STAGE 1: PRE-TRAIN META HEAD
    # -------------------------------------------------
        # STAGE 1: PRE-TRAIN META HEAD
    print("\nSTAGE 1: Pre-training Meta Head")
    for p in model.dec.parameters(): 
        p.requires_grad = False
    opt1 = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-2)
    scaler = amp_GradScaler('cuda')

    for epoch in range(5):
        model.train()
        total_loss = 0
        prog = tqdm(trL, desc=f"S1 | Epoch {epoch+1}/5")
        for eeg, meta, _ in prog:
            eeg, meta = eeg.to(DEVICE), meta.to(DEVICE)
            opt1.zero_grad()
            with amp_autocast('cuda'):
                mp, _ = model(eeg, meta, None, edge_idx, edge_w, stage=1)
                loss = cc(mp[:, :12], meta[:, 0].long()) + 2.0 * oc(mp[:, 12:], meta[:, 1:].float())
            scaler.scale(loss).backward()
            scaler.step(opt1)
            scaler.update()
            total_loss += loss.item()
            prog.set_postfix(loss=f"{loss.item():.3f}")
        print(f"S1 Epoch {epoch+1} | Avg Loss: {total_loss/len(trL):.4f}")
    # -------------------------------------------------
    # STAGE 2: CONTRASTIVE ALIGNMENT
    # -------------------------------------------------
    print("\nSTAGE 2: Contrastive Alignment (CMA)")
    for p in model.dec.parameters(): p.requires_grad = False
    opt2 = AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)

    for epoch in range(10):
        model.train()
        total_cont = 0
        prog = tqdm(trL, desc=f"S2 | Epoch {epoch+1}/10")
        for eeg, meta, _ in prog:
            eeg, meta = eeg.to(DEVICE), meta.to(DEVICE)
            opt2.zero_grad()
            with amp_autocast('cuda'):
                _, loss_cont = model(eeg, meta, None, edge_idx, edge_w, stage=2)
            scaler.scale(loss_cont).backward()
            scaler.step(opt2)
            scaler.update()
            total_cont += loss_cont.item()
            prog.set_postfix(cont_loss=f"{loss_cont.item():.3f}")
        print(f"→ S2 Epoch {epoch+1} | Avg Cont Loss: {total_cont/len(trL):.4f}")

    # -------------------------------------------------
    # STAGE 3: FULL TEXT GENERATION
    # -------------------------------------------------
    print("\nSTAGE 3: Full Text Generation")
    for p in model.parameters(): p.requires_grad = True
    opt3 = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    sched = ReduceLROnPlateau(opt3, 'min', factor=0.5, patience=3)
    tf_sched = np.concatenate([np.ones(30), np.linspace(1.0, 0.0, 30)])
    best = float('inf')

    for epoch in range(1, 31):
        tf = tf_sched[min(epoch-1, len(tf_sched)-1)]
        model.train()
        total_loss = 0
        prog = tqdm(trL, desc=f"S3 | Epoch {epoch}/30 | TF {tf:.2f}")
        for eeg, meta, txt in prog:
            eeg, meta, txt = eeg.to(DEVICE), meta.to(DEVICE), txt.to(DEVICE)
            opt3.zero_grad()
            with amp_autocast('cuda'):
                logits, mp, _ = model(eeg, meta, txt, edge_idx, edge_w, stage=3, tf_ratio=tf)
                loss_text = tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1))
                loss_meta = cc(mp[:, :12], meta[:, 0].long()) + oc(mp[:, 12:], meta[:, 1:].float())
                loss = loss_text + 0.5 * loss_meta
            scaler.scale(loss).backward()
            scaler.step(opt3)
            scaler.update()
            total_loss += loss.item()
            prog.set_postfix(loss=f"{loss.item():.3f}")
        val_loss = 0
        model.eval()
        with torch.no_grad():
            for eeg, meta, txt in vaL:
                eeg, meta, txt = eeg.to(DEVICE), meta.to(DEVICE), txt.to(DEVICE)
                logits, mp, _ = model(eeg, meta, txt, edge_idx, edge_w, stage=3)
                val_loss += (tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1)) + 0.5 * (
                    cc(mp[:, :12], meta[:, 0].long()) + oc(mp[:, 12:], meta[:, 1:].float())
                )).item()
        val_loss /= len(vaL)
        sched.step(val_loss)
        print(f"→ S3 Epoch {epoch} | Train: {total_loss/len(trL):.4f} | Val: {val_loss:.4f}")
        if val_loss < best:
            best = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("   [BEST MODEL SAVED]")

    # -------------------------------------------------
   

CMA-BRAIN2TEXT: NOVEL CONTRASTIVE EEG-TO-TEXT
Dataset: 28000 samples
   EEG shape: (62, 400)
   Meta shape: (91,)
   Text shape: (64,)

STAGE 1: Pre-training Meta Head


S1 | Epoch 1/5:   0%|          | 0/1400 [00:00<?, ?it/s]

S1 Epoch 1 | Avg Loss: 4.5441


S1 | Epoch 2/5:   0%|          | 0/1400 [00:00<?, ?it/s]

S1 Epoch 2 | Avg Loss: 4.4431


...

→ S2 Epoch 9 | Avg Cont Loss: 5.5453


S2 | Epoch 10/10:   0%|          | 0/1400 [00:00<?, ?it/s]

→ S2 Epoch 10 | Avg Cont Loss: 5.5456

STAGE 3: Full Text Generation


S3 | Epoch 1/30 | TF 1.00:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
 # FINAL TEST & GENERATION
    # -------------------------------------------------
    print("\nFINAL TEST & BRAIN-TO-TEXT")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    test_loss, test_cacc = 0, 0
    with torch.no_grad():
        for eeg, meta, txt in teL:
            eeg, meta, txt = eeg.to(DEVICE), meta.to(DEVICE), txt.to(DEVICE)
            logits, mp, _ = model(eeg, meta, txt, edge_idx, edge_w, stage=3)
            test_loss += tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1)).item()
            test_cacc += (mp[:, :12].argmax(1) == meta[:, 0].long()).float().mean().item()
    print(f"TEST → Loss: {test_loss/len(teL):.4f} | ColorAcc: {test_cacc/len(teL):.3f}")

    print("\nSAMPLE BRAIN-TO-TEXT:")
    with torch.no_grad():
        for i, (eeg, _, txt) in enumerate(teL):
            if i >= 3: break
            pred = generate(model, eeg[0:1], edge_idx, edge_w, tokenizer)
            true = tokenizer.decode(txt[0].tolist(), skip_special_tokens=True)
            print(f"EEG → {pred}")
            print(f"True: {true}\n")

In [ ]:
#No meta

In [1]:
# ------------------------------------------------------------
# PURE EEG-TO-TEXT: NO MODE COLLAPSE
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader, random_split
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
from torch.amp import autocast as amp_autocast, GradScaler as amp_GradScaler
import time
from evaluate import load as eval_load
from transformers import AutoTokenizer

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
MODEL_SAVE_PATH = 'eeg2text_final.pt'
BATCH_SIZE = 16
TRAIN_PCT = 0.8
VAL_PCT = 0.1
EPOCHS = 50
D_MODEL = 256
TEXT_VOCAB_SIZE = 32000
PAD_ID = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 1. DATASET — ONLY EEG + TEXT
# ------------------------------------------------------------
class EEGTextOnlyDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
            print(f"Loaded {self.n_samples} samples | EEG: {f['eeg'].shape[1:]} | Text: {f['input_ids'].shape[1:]}")

    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        return eeg, text

def collate_fn(batch):
    eeg, text = zip(*batch)
    eeg = torch.nn.utils.rnn.pad_sequence(eeg, batch_first=True, padding_value=0.0)
    text = torch.nn.utils.rnn.pad_sequence(text, batch_first=True, padding_value=PAD_ID)
    return eeg, text

# ------------------------------------------------------------
# 2. GRAPH
# ------------------------------------------------------------
def fully_connected_graph(n_nodes=62):
    idx = torch.combinations(torch.arange(n_nodes), r=2)
    edge_index = torch.cat([idx, idx.flip(1)], dim=0).t().contiguous()
    edge_weight = torch.ones(edge_index.size(1))
    return edge_index.to(DEVICE), edge_weight.to(DEVICE)

# ------------------------------------------------------------
# 3. MODEL (WITH DROPOUT)
# ------------------------------------------------------------
class EEGEncoder(nn.Module):
    def __init__(self, d_model=256, n_layers=6, n_heads=8, dropout=0.3):
        super().__init__()
        self.gcn = GCNConv(62, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=1024,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_emb = nn.Parameter(torch.randn(1, 1000, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_idx, edge_w):
        B, C, T = eeg.shape
        x = eeg.permute(0, 2, 1).reshape(B*T, C)
        x = self.gcn(x, edge_idx, edge_w).view(B, T, -1)
        x = self.dropout(x + self.pos_emb[:, :T, :])
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        return self.transformer(x)

class TextDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=6, n_heads=8, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=1024,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory):
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.dropout(self.emb(tgt))
        x = self.transformer(x, memory, tgt_mask=mask)
        return self.out(x)

class PureEEG2Text(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.enc = EEGEncoder(D_MODEL, dropout=0.3)
        self.dec = TextDecoder(vocab_size, D_MODEL, dropout=0.3)

    def forward(self, eeg, tgt, edge_idx, edge_w, tf_ratio=1.0):
        memory = self.enc(eeg, edge_idx, edge_w)
        if self.training and tf_ratio > 0 and torch.rand(1) < tf_ratio:
            shifted = torch.cat([torch.full((tgt.size(0), 1), 1, device=tgt.device), tgt[:, :-1]], dim=1)
            tgt = shifted
        return self.dec(tgt, memory)

# ------------------------------------------------------------
# 4. BEAM SEARCH (NO MODE COLLAPSE)
# ------------------------------------------------------------
def beam_search(model, eeg, edge_idx, edge_w, tokenizer, beam_size=5, max_len=30):
    model.eval()
    with torch.no_grad():
        bos_id = tokenizer.bos_token_id or 1
        sequences = [[bos_id], 0.0]  # [seq, score]
        for _ in range(max_len):
            all_candidates = []
            for seq, score in sequences:
                input_tensor = torch.tensor([seq], device=eeg.device)
                logits = model(eeg, input_tensor, edge_idx, edge_w, tf_ratio=0.0)
                probs = F.log_softmax(logits[0, -1], dim=-1)
                topk = probs.topk(beam_size)
                for i in range(beam_size):
                    token = topk.indices[i].item()
                    new_score = score + topk.values[i].item()
                    new_seq = seq + [token]
                    all_candidates.append([new_seq, new_score])
            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_size]
            if sequences[0][0][-1] == tokenizer.eos_token_id:
                break
        best_seq = sequences[0][0]
        return tokenizer.decode(best_seq, skip_special_tokens=True)



/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch_cluster/nearest.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  import scipy.cluster


In [3]:
# ------------------------------------------------------------
# 5. MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    print("PURE EEG-TO-TEXT: NO MODE COLLAPSE")
    print("="*50)

    # Data
    ds = EEGTextOnlyDataset(H5_FILE_PATH)
    N = len(ds)
    trN = int(N*TRAIN_PCT); valN = int(N*VAL_PCT); teN = N-trN-valN
    tr, va, te = random_split(ds, [trN, valN, teN], generator=torch.Generator().manual_seed(42))
    trL = DataLoader(tr, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    vaL = DataLoader(va, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    teL = DataLoader(te, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    edge_idx, edge_w = fully_connected_graph()

    # Model
    model = PureEEG2Text(TEXT_VOCAB_SIZE).to(DEVICE)
    opt = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    sched = ReduceLROnPlateau(opt, 'min', factor=0.5, patience=3)
    tc = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)  # ← KEY
    scaler = amp_GradScaler('cuda')

    # Scheduled sampling
    tf_sched = np.concatenate([np.linspace(1.0, 0.1, 40), np.ones(10)*0.1])
    best_val = float('inf')

    # Training
    for epoch in range(1, EPOCHS + 1):
        tf = tf_sched[min(epoch-1, len(tf_sched)-1)]
        model.train()
        total_loss = 0
        prog = tqdm(trL, desc=f"Epoch {epoch:02d} | TF {tf:.2f}")
        for eeg, txt in prog:
            eeg, txt = eeg.to(DEVICE), txt.to(DEVICE)
            opt.zero_grad()
            with amp_autocast('cuda'):
                logits = model(eeg, txt, edge_idx, edge_w, tf_ratio=tf)
                loss = tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1))
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total_loss += loss.item()
            prog.set_postfix(loss=f"{loss.item():.3f}")

        val_loss = 0
        model.eval()
        with torch.no_grad():
            for eeg, txt in vaL:
                eeg, txt = eeg.to(DEVICE), txt.to(DEVICE)
                logits = model(eeg, txt, edge_idx, edge_w, tf_ratio=0.0)
                val_loss += tc(logits.view(-1, TEXT_VOCAB_SIZE), txt.view(-1)).item()
        val_loss /= len(vaL)
        sched.step(val_loss)

        print(f"Epoch {epoch:02d} | Train: {total_loss/len(trL):.4f} | Val: {val_loss:.4f}")
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(" [BEST SAVED]")

    

PURE EEG-TO-TEXT: NO MODE COLLAPSE
Loaded 28000 samples | EEG: (62, 400) | Text: (64,)


Epoch 01 | TF 1.00:   0%|          | 0/1400 [00:00<?, ?it/s]

Epoch 01 | Train: 3.5906 | Val: 9.9129
 [BEST SAVED]


Epoch 02 | TF 0.98:   0%|          | 0/1400 [00:00<?, ?it/s]

Epoch 02 | Train: 2.4938 | Val: 5.3449
 [BEST SAVED]


...

Epoch 06 | Train: 1.9481 | Val: 2.0840


Epoch 07 | TF 0.86:   0%|          | 0/1400 [00:00<?, ?it/s]

Epoch 07 | Train: 1.9151 | Val: 2.3140


Epoch 08 | TF 0.84:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [7]:
# -------------------------------------------------
# 6. EVALUATION (AFTER TRAINING)
# -------------------------------------------------
import time
from evaluate import load as eval_load
from transformers import AutoTokenizer

# Load BLEU metric
bleu_metric = eval_load("bleu")

# Load best model and tokenizer
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)

print("\n" + "="*60)
print("EVALUATION ON TEST SET (BEAM SEARCH)")
print("="*60)

start_time = time.time()

# Beam search generation
def beam_search(model, eeg, edge_idx, edge_w, tokenizer, beam_size=5, max_len=30):
    model.eval()
    with torch.no_grad():
        bos_id = tokenizer.bos_token_id or 1
        sequences = [([bos_id], 0.0)]  # list of (seq, score)
        for _ in range(max_len):
            all_candidates = []
            for seq, score in sequences:
                input_tensor = torch.tensor([seq], device=eeg.device)
                logits = model(eeg, input_tensor, edge_idx, edge_w, tf_ratio=0.0)
                probs = F.log_softmax(logits[0, -1], dim=-1)
                topk = probs.topk(beam_size)
                for i in range(beam_size):
                    token = topk.indices[i].item()
                    new_score = score + topk.values[i].item()
                    new_seq = seq + [token]
                    all_candidates.append((new_seq, new_score))
            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_size]
            if sequences[0][0][-1] == tokenizer.eos_token_id:
                break
        best_seq = sequences[0][0]
        return tokenizer.decode(best_seq, skip_special_tokens=True)

# Generate predictions
predictions_list = []
references_list = []

print("Generating predictions with beam search...")
with torch.no_grad():
    for eeg, txt in tqdm(teL, desc="Eval"):
        eeg = eeg.to(DEVICE)
        for i in range(eeg.size(0)):
            pred = beam_search(model, eeg[i:i+1], edge_idx, edge_w, tokenizer, beam_size=5)
            ref = tokenizer.decode(txt[i].tolist(), skip_special_tokens=True)
            predictions_list.append(pred)
            references_list.append([ref])  # evaluate expects [[ref]]

# Compute BLEU-4
print("Computing BLEU-4...")
bleu_results = bleu_metric.compute(
    predictions=predictions_list,
    references=references_list,
    max_order=4
)
final_bleu = bleu_results['bleu']

# Timing
end_time = time.time()
elapsed = end_time - start_time
formatted_time = f"{int(elapsed // 60):02d}m {int(elapsed % 60):02d}s"

# Print results
print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"Time taken      : {formatted_time}")
print(f"BLEU-4 Score    : {final_bleu:.4f}")
print(f"Test samples    : {len(predictions_list)}")
print("="*60)

# Show 3 sample outputs
print("\nSAMPLE GENERATIONS:")
for i in range(min(3, len(predictions_list))):
    print(f"Pred: {predictions_list[i]}")
    print(f"Ref : {references_list[i][0]}\n")


EVALUATION ON TEST SET (BEAM SEARCH)
Generating predictions with beam search...


Eval:   0%|          | 0/175 [00:00<?, ?it/s]

Computing BLEU-4...

EVALUATION COMPLETE
Time taken      : 33m 28s
BLEU-4 Score    : 0.0301
Test samples    : 2800

SAMPLE GENERATIONS:
Pred: [unused0] a person plays a piano with their hands.
Ref : a vibrant underwater scene with colorful fish swimming around a coral reef.

Pred: [unused0] a person plays a piano with their hands.
Ref : a serene waterfall cascades through lush greenery, creating a tranquil natural scene.

Pred: [unused0] a person plays a piano with their hands.
Ref : fireworks explode in the night sky, creating a dazzling display of colors and light.



In [23]:
import time
from evaluate import load as eval_load
from transformers import AutoTokenizer

bleu_metric = eval_load("bleu")

print("\n" + "="*50)
print("STARTING EVALUATION ON TEST SET")
print("="*50)

start_time = time.time()

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)

predictions_list = []
references_list  = []

print("Generating predictions...")
with torch.no_grad():
    for eeg, txt in tqdm(teL, desc="Eval"):
        eeg = eeg.to(DEVICE)
        for i in range(eeg.size(0)):
            pred = generate(model, eeg[i:i+1], edge_idx, edge_w, tokenizer, max_len=30)
            ref  = tokenizer.decode(txt[i].tolist(), skip_special_tokens=True)

            predictions_list.append(pred)
            references_list.append([ref])

# BLEU
bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list, max_order=4)
final_bleu = bleu_results['bleu']

# Timing
end_time = time.time()
elapsed = end_time - start_time
formatted_time = f"{int(elapsed // 60):02d}m {int(elapsed % 60):02d}s"

# Print
print("\n" + "="*50)
print("EVALUATION DONE")
print(f"Time : {formatted_time}")
print(f"BLEU-4: {final_bleu:.4f}")
print(f"Samples: {len(predictions_list)}")
print("="*50)

# Show 3 examples
print("\nSAMPLES:")
for i in range(min(3, len(predictions_list))):
    print(f"Pred: {predictions_list[i]}")
    print(f"Ref : {references_list[i][0]}\n")


STARTING EVALUATION ON TEST SET
Generating predictions...


Eval:   0%|          | 0/175 [00:00<?, ?it/s]


EVALUATION DONE
Time : 07m 25s
BLEU-4: 0.0257
Samples: 2800

SAMPLES:
Pred: [unused0] a person with a black and white white white in the background.
Ref : a vibrant underwater scene with colorful fish swimming around a coral reef.

Pred: [unused0] a person with a black and white white white in the background.
Ref : a serene waterfall cascades through lush greenery, creating a tranquil natural scene.

Pred: [unused0] a person with a black and white white white in the background.
Ref : fireworks explode in the night sky, creating a dazzling display of colors and light.



In [8]:
#Screw this wow